# Experimental data visualization and ROI cropping

Use the center-crop slider workflow to inspect experimental rocking-curve data, click the dislocation core, crop raw ROIs, resize them to the neural-network input shape, add scale bars, and save the extracted images.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from bv_repro.data import load_edf_stack


In [ ]:
# Data location
DATA_ROOT = Path("/data/projects/engage_id03/cross_slip_data/Al_sample_5_tick_revolution_70/Al_sample_5_tick_revolution_70")

# Rocking-frame settings
START_FRAME = 4                 # integrate from the 4th image onward
SHOW_FRAME = 4                  # show this single rocking image for the non-integrated ROI
APPLY_VERTICAL_STRETCH = True   # stretch vertically by 1/tan(18 deg), same as previous workflow
STRETCH_ANGLE_DEG = 18.0
RAW_ROI_SHAPE = (60, 40)        # raw ROI around clicked core: (height, width) = 60 x 40 px
TARGET_SHAPE = (510, 170)       # resized neural-network input shape: (height, width)
SUBTRACT_DISPLAY_MEAN = True    # subtract mean from the resized 4th-frame ROI


# Layer order shown in the slider
LAYER_ORDER = [f"Z_local_rocking_2x_{i:02d}" for i in range(11)]

# Display settings
DISPLAY_LOG = True
DISPLAY_PERCENTILES = (1.0, 99.5)
CMAP = "viridis"

# Scale bar settings. Change this if your calibrated pixel size is different.
PIXEL_SIZE_UM = 0.4
SCALE_BAR_UM = 5.0
SCALE_BAR_COLOR = "red"

# Optional saving
SAVE_FIGURES = True
OUTPUT_DIR = Path("fixed_roi_scalebar_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def subtract_image_mean(image, clip_negative=True):
    image = np.asarray(image, dtype=np.float32)
    corrected = image - float(image.mean())
    if clip_negative:
        corrected = np.clip(corrected, a_min=0.0, a_max=None)
    return corrected


def crop_xyxy(image, roi_xyxy):
    x0, y0, x1, y1 = [int(v) for v in roi_xyxy]
    return image[y0:y1, x0:x1]


def integrate_roi(stack, roi_xyxy, start_frame=4):
    roi_stack = np.stack([crop_xyxy(frame, roi_xyxy) for frame in stack[start_frame:]], axis=0)
    return roi_stack.sum(axis=0)


def resize_to_shape(image, target_shape=(510, 170)):
    image = np.asarray(image, dtype=np.float32)
    target_h, target_w = target_shape
    zoom_y = target_h / image.shape[0]
    zoom_x = target_w / image.shape[1]
    try:
        from scipy.ndimage import zoom
    except ImportError as exc:
        raise ImportError("Resizing the ROI to TARGET_SHAPE requires scipy.ndimage.zoom.") from exc
    return zoom(image, zoom=(zoom_y, zoom_x), order=1).astype(np.float32)


def effective_resized_x_pixel_size_um(roi_xyxy, target_width, original_pixel_size_um=0.4):
    x0, y0, x1, y1 = [int(v) for v in roi_xyxy]
    original_width_px = max(1, x1 - x0)
    physical_width_um = original_width_px * original_pixel_size_um
    return physical_width_um / target_width


def display_image(image, log_scale=True, percentiles=(1.0, 99.5)):
    arr = np.asarray(image, dtype=np.float32)
    if log_scale:
        arr = np.log1p(np.clip(arr, a_min=0.0, a_max=None))
    vmin, vmax = np.percentile(arr, percentiles)
    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0
    return np.clip(arr, vmin, vmax), vmin, vmax


def add_scale_bar(ax, image_shape, pixel_size_um=0.4, bar_um=5.0, color="red"):
    h, w = image_shape[:2]
    bar_px = bar_um / pixel_size_um
    bar_px = min(bar_px, w * 0.65)

    margin_x = max(3, int(0.08 * w))
    margin_y = max(3, int(0.08 * h))
    thickness = max(2, int(0.012 * max(h, w)))

    x0 = w - margin_x - bar_px
    y0 = h - margin_y

    ax.add_patch(Rectangle((x0, y0), bar_px, thickness, color=color, lw=0))
    ax.text(
        x0 + bar_px / 2,
        y0 - 1.8 * thickness,
        f"{bar_um:g} $\\mu$m",
        color=color,
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )


def show_roi_pair(layer_name, roi_frame, integrated_roi, pixel_size_um=None, save=False):
    if pixel_size_um is None:
        pixel_size_um = PIXEL_SIZE_UM
    fig, axes = plt.subplots(1, 2, figsize=(5.5, 4.0), constrained_layout=True)
    for ax, image, title in [
        (axes[0], roi_frame, f"{layer_name}\nframe {SHOW_FRAME}"),
        (axes[1], integrated_roi, f"{layer_name}\nintegrated {START_FRAME}-end"),
    ]:
        disp, vmin, vmax = display_image(
            image,
            log_scale=DISPLAY_LOG,
            percentiles=DISPLAY_PERCENTILES,
        )
        ax.imshow(disp, cmap=CMAP, vmin=vmin, vmax=vmax, origin="upper")
        add_scale_bar(
            ax,
            disp.shape,
            pixel_size_um=pixel_size_um,
            bar_um=SCALE_BAR_UM,
            color=SCALE_BAR_COLOR,
        )
        ax.set_title(title, fontsize=9)
        ax.set_axis_off()

    if save:
        out = OUTPUT_DIR / f"{layer_name}_roi_and_integrated.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print("saved", out)
    plt.show()


## Slider picker

The displayed image is the central `1000 x 1000` crop from the 4th rocking-curve step. Click the dislocation core; the notebook stores that centre and draws a red bounding box with the neural-network input size `(height=510, width=170)`.


In [ ]:
# Run this first if the click event does not work in your notebook:
# %matplotlib widget

import ipywidgets as widgets
from IPython.display import display

CENTER_PREVIEW_SHAPE = (1000, 1000)  # (height, width) shown in the slider picker


def center_crop_bounds(image_shape, crop_shape=(1000, 1000)):
    h, w = image_shape[:2]
    crop_h, crop_w = crop_shape
    crop_h = min(crop_h, h)
    crop_w = min(crop_w, w)
    px0 = (w - crop_w) // 2
    py0 = (h - crop_h) // 2
    px1 = px0 + crop_w
    py1 = py0 + crop_h
    return px0, py0, px1, py1


def crop_centered_with_padding(image, center_xy, output_shape=(60, 40), fill_value=0.0):
    image = np.asarray(image, dtype=np.float32)
    out_h, out_w = output_shape
    h, w = image.shape[:2]
    cx, cy = center_xy
    cx = int(round(cx))
    cy = int(round(cy))

    x0 = cx - out_w // 2
    y0 = cy - out_h // 2
    x1 = x0 + out_w
    y1 = y0 + out_h

    src_x0 = max(0, x0)
    src_y0 = max(0, y0)
    src_x1 = min(w, x1)
    src_y1 = min(h, y1)

    dst_x0 = src_x0 - x0
    dst_y0 = src_y0 - y0
    dst_x1 = dst_x0 + (src_x1 - src_x0)
    dst_y1 = dst_y0 + (src_y1 - src_y0)

    out = np.full((out_h, out_w), fill_value, dtype=np.float32)
    out[dst_y0:dst_y1, dst_x0:dst_x1] = image[src_y0:src_y1, src_x0:src_x1]
    bounds = (x0, y0, x1, y1)
    return out, bounds


def bbox_from_center(center_xy, shape=(60, 40)):
    out_h, out_w = shape
    cx, cy = center_xy
    cx = int(round(cx))
    cy = int(round(cy))
    x0 = cx - out_w // 2
    y0 = cy - out_h // 2
    x1 = x0 + out_w
    y1 = y0 + out_h
    return x0, y0, x1, y1


def clamp_bbox_for_display(bbox, image_shape):
    h, w = image_shape[:2]
    x0, y0, x1, y1 = bbox
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)


def preview_bounds_for_layer(layer_name, image_shape):
    return center_crop_bounds(image_shape, crop_shape=CENTER_PREVIEW_SHAPE)


def show_single_scaled_image(image, title, output_path=None, pixel_size_um=0.4):
    disp, vmin, vmax = display_image(
        image,
        log_scale=DISPLAY_LOG,
        percentiles=DISPLAY_PERCENTILES,
    )
    fig, ax = plt.subplots(figsize=(3.2, 8.0), constrained_layout=True)
    ax.imshow(disp, cmap=CMAP, vmin=vmin, vmax=vmax, origin="upper")
    add_scale_bar(
        ax,
        disp.shape,
        pixel_size_um=pixel_size_um,
        bar_um=SCALE_BAR_UM,
        color=SCALE_BAR_COLOR,
    )
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print("saved", output_path)
    plt.show()


In [ ]:
# Load only the 4th rocking-curve image for each layer for the interactive picker.
picker_frame_images = {}

for layer_name in LAYER_ORDER:
    layer_dir = DATA_ROOT / layer_name
    stack, files = load_edf_stack(
        layer_dir,
        angle_deg=STRETCH_ANGLE_DEG,
        apply_stretch=APPLY_VERTICAL_STRETCH,
    )
    picker_frame_images[layer_name] = stack[SHOW_FRAME]
    print(layer_name, picker_frame_images[layer_name].shape)


In [ ]:
class DislocationCenterSliderPicker:
    def __init__(self, images, layer_order, raw_roi_shape=(60, 40)):
        self.images = images
        self.layer_order = list(layer_order)
        self.raw_roi_shape = raw_roi_shape
        self.centers = {}
        self.boxes = {}
        self.preview_bounds = {}
        self.fig = None
        self.ax = None
        self.cid = None

        self.layer_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=len(self.layer_order) - 1,
            step=1,
            description="Layer",
            continuous_update=False,
            layout=widgets.Layout(width="70%"),
        )
        self.zoom_slider = widgets.IntSlider(
            value=CENTER_PREVIEW_SHAPE[0],
            min=250,
            max=1600,
            step=50,
            description="View px",
            continuous_update=False,
            layout=widgets.Layout(width="70%"),
        )
        self.low_slider = widgets.FloatSlider(
            value=DISPLAY_PERCENTILES[0],
            min=0.0,
            max=20.0,
            step=0.5,
            description="Low %",
            continuous_update=False,
            layout=widgets.Layout(width="70%"),
        )
        self.high_slider = widgets.FloatSlider(
            value=DISPLAY_PERCENTILES[1],
            min=80.0,
            max=100.0,
            step=0.1,
            description="High %",
            continuous_update=False,
            layout=widgets.Layout(width="70%"),
        )
        self.log_toggle = widgets.Checkbox(value=DISPLAY_LOG, description="log")

        self.prev_button = widgets.Button(description="Prev", icon="arrow-left")
        self.next_button = widgets.Button(description="Next", icon="arrow-right")
        self.clear_button = widgets.Button(description="Clear current", icon="trash")

        self.x_text = widgets.FloatText(description="x", layout=widgets.Layout(width="180px"))
        self.y_text = widgets.FloatText(description="y", layout=widgets.Layout(width="180px"))
        self.use_xy_button = widgets.Button(description="Use typed centre", icon="crosshairs")

        self.status = widgets.HTML()
        self.output = widgets.Output()

        for widget in [self.layer_slider, self.zoom_slider, self.low_slider, self.high_slider, self.log_toggle]:
            widget.observe(self.on_control_change, names="value")
        self.prev_button.on_click(self.on_prev)
        self.next_button.on_click(self.on_next)
        self.clear_button.on_click(self.on_clear_current)
        self.use_xy_button.on_click(self.on_use_typed_center)

    def current_layer(self):
        return self.layer_order[int(self.layer_slider.value)]

    def current_crop_shape(self):
        size = int(self.zoom_slider.value)
        return size, size

    def current_preview(self):
        layer_name = self.current_layer()
        image = self.images[layer_name]
        px0, py0, px1, py1 = center_crop_bounds(image.shape, crop_shape=self.current_crop_shape())
        self.preview_bounds[layer_name] = (px0, py0, px1, py1)
        return image[py0:py1, px0:px1], (px0, py0, px1, py1)

    def current_display_percentiles(self):
        low = float(self.low_slider.value)
        high = float(self.high_slider.value)
        if high <= low:
            high = min(100.0, low + 0.1)
        return low, high

    def draw(self):
        layer_name = self.current_layer()
        preview, (px0, py0, px1, py1) = self.current_preview()
        low, high = self.current_display_percentiles()
        disp, vmin, vmax = display_image(
            preview,
            log_scale=bool(self.log_toggle.value),
            percentiles=(low, high),
        )

        self.output.clear_output(wait=True)
        with self.output:
            self.fig, self.ax = plt.subplots(figsize=(7, 7))
            self.ax.imshow(disp, cmap=CMAP, vmin=vmin, vmax=vmax, origin="upper")
            self.ax.set_title(
                f"{layer_name}: centre {disp.shape[1]} x {disp.shape[0]} view, click core; red box = raw 40 x 60 px",
                fontsize=11,
            )
            self.ax.set_axis_off()
            add_scale_bar(
                self.ax,
                disp.shape,
                pixel_size_um=PIXEL_SIZE_UM,
                bar_um=SCALE_BAR_UM,
                color=SCALE_BAR_COLOR,
            )

            if layer_name in self.centers:
                cx, cy = self.centers[layer_name]
                local_x = cx - px0
                local_y = cy - py0
                x0, y0, x1, y1 = self.boxes[layer_name]
                self.ax.plot(local_x, local_y, marker="+", color="red", markersize=12, mew=2)
                self.ax.add_patch(
                    Rectangle(
                        (x0 - px0, y0 - py0),
                        x1 - x0,
                        y1 - y0,
                        fill=False,
                        edgecolor="red",
                        linewidth=1.0,
                    )
                )

            self.cid = self.fig.canvas.mpl_connect("button_press_event", self.on_click)
            plt.show()

        selected = len(self.centers)
        self.status.value = (
            f"<b>{layer_name}</b> | selected {selected}/{len(self.layer_order)} layers | "
            f"full-image preview bounds: x={px0}:{px1}, y={py0}:{py1}"
        )

    def set_center(self, layer_name, center):
        self.centers[layer_name] = (float(center[0]), float(center[1]))
        self.boxes[layer_name] = bbox_from_center(center, self.raw_roi_shape)
        self.x_text.value = float(center[0])
        self.y_text.value = float(center[1])

    def on_click(self, event):
        if self.ax is None or event.inaxes != self.ax or event.xdata is None or event.ydata is None:
            return
        layer_name = self.current_layer()
        px0, py0, px1, py1 = self.preview_bounds[layer_name]
        center = (px0 + float(event.xdata), py0 + float(event.ydata))
        self.set_center(layer_name, center)
        self.draw()

    def on_control_change(self, change):
        self.draw()

    def on_prev(self, button):
        self.layer_slider.value = max(self.layer_slider.min, self.layer_slider.value - 1)

    def on_next(self, button):
        self.layer_slider.value = min(self.layer_slider.max, self.layer_slider.value + 1)

    def on_clear_current(self, button):
        layer_name = self.current_layer()
        self.centers.pop(layer_name, None)
        self.boxes.pop(layer_name, None)
        self.draw()

    def on_use_typed_center(self, button):
        layer_name = self.current_layer()
        self.set_center(layer_name, (self.x_text.value, self.y_text.value))
        self.draw()

    def get_centers(self):
        return dict(self.centers)

    def get_boxes(self):
        return dict(self.boxes)

    def show(self):
        controls = widgets.VBox([
            widgets.HBox([self.prev_button, self.next_button, self.clear_button]),
            self.layer_slider,
            self.zoom_slider,
            widgets.HBox([self.low_slider, self.high_slider, self.log_toggle]),
            widgets.HBox([self.x_text, self.y_text, self.use_xy_button]),
            self.status,
        ])
        display(widgets.VBox([controls, self.output]))
        self.draw()
        return self


center_picker = DislocationCenterSliderPicker(
    picker_frame_images,
    LAYER_ORDER,
    raw_roi_shape=RAW_ROI_SHAPE,
).show()


In [ ]:
# After selecting the centres with the slider, run this cell.
slider_centers = center_picker.get_centers()
slider_boxes = center_picker.get_boxes()

for layer_name in LAYER_ORDER:
    if layer_name not in slider_boxes:
        print("missing", layer_name)
        continue
    print(layer_name, "center", tuple(round(v, 2) for v in slider_centers[layer_name]), "bbox", slider_boxes[layer_name])


In [ ]:
# Crop raw 40x60 boxes around clicked centres from the non-integrated 4th frame,
# resize them to 510x170, then plot each image separately with a scale bar.
slider_roi_frames = {}

for layer_name in LAYER_ORDER:
    if layer_name not in slider_centers:
        print("Skipping, no centre selected:", layer_name)
        continue

    layer_dir = DATA_ROOT / layer_name
    stack, files = load_edf_stack(
        layer_dir,
        angle_deg=STRETCH_ANGLE_DEG,
        apply_stretch=APPLY_VERTICAL_STRETCH,
    )

    frame_crop_raw, frame_bounds = crop_centered_with_padding(
        stack[SHOW_FRAME],
        slider_centers[layer_name],
        output_shape=RAW_ROI_SHAPE,
    )

    frame_crop = resize_to_shape(frame_crop_raw, TARGET_SHAPE)

    if SUBTRACT_DISPLAY_MEAN:
        frame_crop = subtract_image_mean(frame_crop, clip_negative=True)

    slider_roi_frames[layer_name] = frame_crop

    frame_out = OUTPUT_DIR / f"{layer_name}_slider_center_frame_{SHOW_FRAME:02d}_raw40x60_resized510x170.png" if SAVE_FIGURES else None

    show_single_scaled_image(
        frame_crop,
        f"{layer_name} frame {SHOW_FRAME} ROI",
        output_path=frame_out,
        pixel_size_um=(RAW_ROI_SHAPE[1] * PIXEL_SIZE_UM / TARGET_SHAPE[1]),
    )


In [ ]:
# Save the slider-selected non-integrated crops and bounding boxes.
slider_boxes_path = OUTPUT_DIR / "slider_selected_centres_raw40x60_boxes.csv"
with slider_boxes_path.open("w", encoding="utf-8") as f:
    f.write("layer,x_center,y_center,x0,y0,x1,y1\n")
    for layer_name in LAYER_ORDER:
        if layer_name not in slider_centers:
            continue
        x, y = slider_centers[layer_name]
        x0, y0, x1, y1 = slider_boxes[layer_name]
        f.write(f"{layer_name},{x:.6f},{y:.6f},{x0},{y0},{x1},{y1}\n")

for layer_name in LAYER_ORDER:
    if layer_name not in slider_roi_frames:
        continue
    np.save(
        OUTPUT_DIR / f"{layer_name}_slider_center_frame_{SHOW_FRAME:02d}_raw40x60_resized510x170.npy",
        slider_roi_frames[layer_name],
    )

print("saved slider boxes:", slider_boxes_path.resolve())
